# 🧠 Cheeni Capability 1: Intent Classification (Amazon MASSIVE Benchmark)
### Fine-Tuning `microsoft/deberta-v3-large` on Amazon Alexa Voice Utterances (English & Hindi)

**Dataset**: Official Amazon MASSIVE (`en-US` + `hi-IN` subsets) from Amazon Science S3  
**Target Hugging Face Model**: `wraith-klu/cheeni-intent-deberta`  
**Compute**: Google Colab (A100 or T4 GPU) / Kaggle (P100 / A100)

In [ ]:
# 1. Install Hugging Face & ML libraries
!pip install -q transformers datasets evaluate accelerate scikit-learn huggingface_hub

In [ ]:
# 2. Authenticate with your Hugging Face account
from huggingface_hub import login
import os

# Paste your write token from https://huggingface.co/settings/tokens
HF_TOKEN = os.getenv("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
if HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)
else:
    login()

In [ ]:
# 3. Download Official Amazon MASSIVE JSONL (Direct from Amazon S3 - No Script Errors!)
!wget -q -nc https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz
!tar -xzf amazon-massive-dataset-1.0.tar.gz

from datasets import load_dataset, concatenate_datasets

print("📥 Loading English (en-US) and Indian Hindi (hi-IN) JSONL files...")
massive_en = load_dataset("json", data_files="1.0/data/en-US.jsonl", split="train")
massive_hi = load_dataset("json", data_files="1.0/data/hi-IN.jsonl", split="train")

# Combine English and Hindi for bilingual voice comprehension
full_dataset = concatenate_datasets([massive_en, massive_hi])

# Split into Train (train partition) and Validation (dev partition)
train_dataset = full_dataset.filter(lambda x: x["partition"] == "train")
eval_dataset = full_dataset.filter(lambda x: x["partition"] == "dev")

print(f"✅ Successfully loaded: {len(train_dataset):,} training samples, {len(eval_dataset):,} validation samples.")
print(f"Sample utterance: '{train_dataset[0]['utt']}' -> Intent: '{train_dataset[0]['intent']}'")

In [ ]:
# 4. Build Label Mappings (60 Intent Classes)
unique_intents = sorted(list(set(train_dataset["intent"])))
num_classes = len(unique_intents)

id2label = {i: label for i, label in enumerate(unique_intents)}
label2id = {label: i for i, label in enumerate(unique_intents)}

print(f"Total unique intents: {num_classes}")

# Map label strings to integers
def encode_labels(example):
    example["label"] = label2id[example["intent"]]
    return example

train_dataset = train_dataset.map(encode_labels)
eval_dataset = eval_dataset.map(encode_labels)

In [ ]:
# 5. Tokenize using DeBERTa-v3-large
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-large"
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["utt"], truncation=True, max_length=128, padding="max_length")

train_encoded = train_dataset.map(tokenize_fn, batched=True)
eval_encoded = eval_dataset.map(tokenize_fn, batched=True)

In [ ]:
# 6. Metrics evaluation (Accuracy & Macro F1)
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

In [ ]:
# 7. Initialize Model & Train with Trainer
# DeBERTa-v3 uses disentangled attention which is unstable in FP16, so we use FP32 (fp16=False) + gradient clipping.
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./cheeni-deberta-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=eval_encoded,
    compute_metrics=compute_metrics,
)

print("🚀 Starting Training on GPU...")
trainer.train()

In [ ]:
# 8. Test the Fine-Tuned Model with Voice Prompts
test_utterances = [
    "open spotify and play party music",
    "kripya kal ka mausam batao",
    "set a reminder for my assignment at five pm",
    "turn off the lights in the bedroom",
    "what is on my schedule today"
]

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for text in test_utterances:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        idx = torch.argmax(probs).item()
        print(f"Query: '{text}' -> Intent: {id2label[idx]} ({probs[idx]:.2%})")

In [ ]:
# 9. Push Directly to your Hugging Face Hub (wraith-klu/cheeni-intent-deberta)
HF_REPO_ID = "wraith-klu/cheeni-intent-deberta"

print(f"Pushing trained model to https://huggingface.co/{HF_REPO_ID}...")
model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"🎉 SUCCESS! Model is live on Hugging Face: https://huggingface.co/{HF_REPO_ID}")